# LAB 6 

### Task I

Un desarrollador junior de su equipo sugiere: "Para detectar con mayor precisión las texturas de las
hojas enfermas, deberíamos construir una red secuencial clásica (tipo VGG) pero de 150 capas. Más
profundo siempre es mejor". Como líder técnico, explíquele argumentativamente por qué esta red
fracasará estrepitosamente en el entrenamiento (mencionando el fenómeno de degradación y el
desvanecimiento del gradiente). Luego, justifique cómo la adición estructural de las conexiones
residuales (𝐹(𝑥) + 𝑥) de ResNet rescata el proyecto, haciendo viable entrenar redes ultra-profundas
sin colapsar.

# Justificación arquitectónica para evitar una red tipo VGG de 150 capas

Le diría, 

Tu intuición de usar una red más profunda para capturar mejor las texturas de las hojas es normal pensar q entre mas capas, mas sabe.

Pero, construir una **una VGG de 150 capas** introduce problemas de optimización durante el entrenamiento con *backpropagation*.

En particular aparecen dos fenómenos:

- **Vanishing Gradient**
- **Degradation Problem**

Estos problemas hacen que una red de esa profundidad no entrene correctamente.

---

## Vanishing Gradient

Durante el entrenamiento utilizamos *backpropagation* para actualizar los pesos.

El gradiente se propaga hacia atrás multiplicándose a través de muchas capas. Si esos valores son menores que 1, el gradiente se reduce de forma exponencial.

Por ejemplo:

$$
0.9^{150} \approx 1.37 \times 10^{-7}
$$

Esto significa que el gradiente que llega a las primeras capas es **casi cero**.

### Consecuencias

- El aprendizaje en las capas cercanas al input se vuelve mínimo.
- Los pesos de esas capas casi no se actualizan.
- El entrenamiento pierde efectividad.

En una red secuencial de **150 capas**, este efecto aparece con facilidad, especialmente considerando que el dataset de enfermedades de hojas de mango no es grande.

---

## Degradation Problem

Aunque el problema del gradiente se reduzca, aparece otro fenómeno importante: **el problema de degradación**.

Si agregas más capas a una red que ya funciona, el modelo debería poder **copiar la solución anterior**. Esto ocurriría si las capas nuevas aprendieran una función identidad.

Sin embargo, en la práctica ocurre algo como:

| Arquitectura | Error de entrenamiento |
|---------------|-----------------------|
| 20 capas | Bajo |
| 56 capas | Más alto |

Esto indica que el modelo **no logra optimizarse correctamente**.

No se trata de **overfitting**, sino de un problema de optimización. Las redes profundas tienen dificultad para aprender transformaciones cercanas a la identidad.

---

## ResNet

La idea es cambiar lo que aprende cada bloque de la red.

En una red tradicional, las capas intentan aprender una función completa:

$$
H(x)
$$

ResNet propone aprender una **función residual**:

$$
F(x) = H(x) - x
$$

La salida del bloque se calcula como:

$$
y = F(x) + x
$$

Este mecanismo introduce una **skip connection**, donde la entrada del bloque se suma directamente a su salida.

---

## Por qué las conexiones residuales funcionan

Las conexiones residuales ayudan por dos razones principales.

### 1. Mejor flujo del gradiente

El gradiente puede propagarse por dos caminos:

- a través de las capas convolucionales
- a través de la conexión de identidad

Esto reduce el problema de **vanishing gradient** y permite entrenar redes profundas.

---

### 2. Facilitan aprender la identidad

Si un bloque adicional no mejora la representación, la red puede aprender:

$$
F(x) = 0
$$

lo que produce:

$$
y = x
$$

De esta forma, las capas extra **no degradan el rendimiento**, resolviendo el **degradation problem**.

---

## Aplicación al proyecto AgriTech

En el contexto de nuestra aplicación para **detección de enfermedades en hojas de mango**, estas decisiones arquitectónicas son importantes.

Nuestro escenario tiene varias restricciones:

- Dataset limitado  
- Texturas complejas en hojas  
- Recursos de entrenamiento limitados  
- Inferencia en **teléfonos Android de gama baja**  
- Aplicación **100% offline (Edge AI)**  

Una arquitectura basada en **ResNet (por ejemplo ResNet-18 o ResNet-50)** ofrece ventajas:

- Podemos entrenar redes profundas de forma estable.
- Captura patrones visuales en texturas de hojas enfermas.
- Reduce problemas de optimización en redes profundas.
- Produce modelos que luego pueden **optimizarse (quantization o pruning)** para ejecutarse en dispositivos móviles.

Por estas razones, en lugar de construir una **VGG secuencial de 150 capas**, una arquitectura basada en **bloques residuales** es una decisión alineada con las necesidades del proyecto AgriTech.

### Inciso 2

Las enfermedades en las hojas de mango son visualmente heterogéneas: La Antracnosis se
presenta como puntos negros diminutos, mientras que el Moho Polvoriento cubre áreas enormes de
la hoja. Analice cómo la topología en paralelo del módulo Inception (usando filtros 3x3 y 5x5
simultáneamente) es ideal para este problema biológico en particular. Además, desde una
perspectiva de costos de infraestructura (uso de GPUs en AWS o Google Cloud), explique cómo la
inserción estratégica de convoluciones de 1x1 evita la explosión de la dimensionalidad y salva el
presupuesto mensual de la startup.




# Uso de módulos Inception para detección de enfermedades en hojas de mango

Las enfermedades en hojas de mango presentan **heterogeneidad visual significativa**.

Por ejemplo:

- **Antracnosis** genera **puntos negros diminutos** distribuidos en la hoja.
- **Moho Polvoriento** produce **regiones extensas cubiertas por una capa blanca**.

Esta variabilidad implica que **los patrones relevantes aparecen en múltiples escalas espaciales**. Un modelo que utilice únicamente un tamaño de filtro corre el riesgo de perder información crítica: filtros pequeños capturan detalles finos pero ignoran patrones amplios; filtros grandes detectan estructuras globales pero diluyen texturas pequeñas.

---

El número aproximado de operaciones de una convolución es:

$$
\text{Costo} \approx H \times W \times C_{in} \times C_{out} \times K^2
$$

donde:

- $H, W$ = dimensiones espaciales  
- $C_{in}$ = canales de entrada  
- $C_{out}$ = canales de salida  
- $K$ = tamaño del kernel  

Filtros **5×5** incrementan el costo debido al término $K^2$.

En infraestructura cloud (AWS o Google Cloud), esto se traduce en:

- mayor tiempo de entrenamiento
- mayor uso de memoria de GPU
- incremento del costo mensual de cómputo

---

## Rol de las convoluciones 1×1

La arquitectura Inception introduce **convoluciones 1×1** antes de aplicar filtros grandes.

Estas capas **proyecciones de reducción de dimensionalidad**.  
Su objetivo es disminuir $C_{in}$ antes de aplicar operaciones costosas.

Ejemplo conceptual:

Entrada:

$$
C_{in} = 256
$$

Aplicar 5×5 directo:

$$
256 \times 5 \times 5
$$

Con reducción previa:

$$
256 \xrightarrow[]{1\times1} 64
$$

Luego:

$$
64 \times 5 \times 5
$$

El número de operaciones disminuye de forma significativa.

---

## Impacto en costos de infraestructura

La reducción de canales mediante **1×1 convolutions** produce beneficios directos:

- menor uso de memoria en GPU  
- menor tiempo de entrenamiento  
- menor consumo de recursos cloud  

Para una startup AgriTech con presupuesto limitado, esto permite:

- entrenar modelos complejos sin necesidad de GPUs de alta gama  
- reducir horas de cómputo en AWS o Google Cloud  
- mantener un pipeline de entrenamiento sostenible

---

## Aplicación

La combinación de **módulos Inception** y **reducción dimensional con 1×1 convolutions** ofrece una arquitectura adecuada para el sistema de diagnóstico de enfermedades en hojas de mango:

- análisis de **patrones visuales en múltiples escalas**
- detección simultánea de **micro-lesiones y regiones extensas**
- control del costo computacional durante el entrenamiento

Esto permite desarrollar un modelo robusto que posteriormente puede optimizarse para **Edge AI en dispositivos Android de gama baja**, cumpliendo con el requisito de inferencia **100% offline** en las fincas agrícolas.

### Inciso 3

El cliente final (el agricultor) usará la aplicación en un teléfono Android de gama baja en medio del
campo, sin conexión a internet. Sabemos que MobileNet logra esta eficiencia gracias a la Depthwise
Separable Convolution. Describa brevemente cómo esta convolución divide el trabajo (filtrado
espacial vs. combinación de canales). Sin embargo, en ingeniería no hay soluciones mágicas, todo tiene un precio: ¿Qué capacidad matemática o expresividad de la red se está sacrificando al separar
estos dos pasos en comparación con una convolución estándar? Analice críticamente por qué, como
director del proyecto, usted acepta pagar ese "costo" en este escenario comercial.

# Depthwise Separable Convolution en MobileNet

## División del trabajo en la convolución separable

Una convolución estándar realiza simultáneamente dos tareas:

1. **filtrado espacial**
2. **combinación de canales**

Si la entrada tiene $C_{in}$ canales y el kernel es $K \times K$, cada filtro tiene tamaño:

$$
K \times K \times C_{in}
$$

El costo computacional aproximado es:

$$
H \times W \times C_{in} \times C_{out} \times K^2
$$

MobileNet divide esta operación en dos pasos.

### Depthwise Convolution

Se aplica **un filtro espacial por cada canal**:

$$
K \times K \times 1
$$

Esta etapa realiza **filtrado espacial**, detectando bordes o texturas dentro de cada canal.

### Pointwise Convolution

Luego se aplica una convolución **1×1** que combina los canales generados:

$$
1 \times 1 \times C_{in}
$$

Esta operación construye nuevas representaciones mediante **mezcla de canales**.

El costo total se reduce a:

$$
H \times W \times C_{in} \times K^2
+
H \times W \times C_{in} \times C_{out}
$$

lo cual es considerablemente menor que la convolución estándar.

---

## Costo en capacidad representacional

La separación de estas operaciones reduce la **expresividad matemática del modelo**.

En una convolución estándar, cada filtro aprende patrones espaciales considerando **interacciones entre todos los canales simultáneamente**.  
En la convolución separable, el filtrado espacial ocurre **de forma independiente por canal**, y la combinación se realiza posteriormente mediante una operación lineal.

Esto equivale a una **factorización del filtro tridimensional**, lo que limita la capacidad para modelar ciertas correlaciones espaciales complejas entre canales.

---

## Justificación en el contexto del proyecto

Como director del proyecto, aceptar esta reducción de capacidad es razonable debido a las restricciones del sistema:

- inferencia **offline**
- hardware **móvil limitado**
- necesidad de **baja latencia**

Depthwise Separable Convolution reduce drásticamente el costo computacional y permite ejecutar el modelo en dispositivos Android de gama baja.  
Aunque existe una pérdida de expresividad, MobileNet mantiene suficiente capacidad para detectar patrones visuales relevantes en enfermedades de hojas de mango, lo cual hace viable el producto final.